# 🧬 Python Metaclasses: Class Creation and Modification

Welcome to **Python Mastery**! This notebook covers **metaclasses** - one of Python's most powerful and advanced features that allows you to control class creation and behavior.

## 📋 What You'll Learn
- Understanding metaclasses and their purpose
- Creating custom metaclasses
- The class creation process
- Method resolution order (MRO)
- Practical metaclass use cases
- Metaclass inheritance and composition
- Debugging metaclass code

## 🎯 Learning Objectives
By the end of this notebook, you'll be able to:
- Understand how metaclasses work internally
- Create custom metaclasses for specific behaviors
- Modify class creation dynamically
- Implement design patterns using metaclasses
- Debug complex metaclass inheritance
- Apply metaclasses in real-world scenarios
- Understand when and why to use metaclasses

---

## 🏗️ Understanding Metaclasses

Metaclasses are classes that create classes. In Python, everything is an object, including classes themselves.

In [ ]:
# Everything is an object in Python
print("=== EVERYTHING IS AN OBJECT ===")

# Regular objects
class Person:
    def __init__(self, name):
        self.name = name

person = Person("Alice")
print(f"person is instance of Person: {isinstance(person, Person)}")
print(f"Person is instance of type: {isinstance(Person, type)}")
print(f"type is instance of type: {isinstance(type, type)}")

# Classes are objects created by metaclasses
print(f"\nPerson.__class__: {Person.__class__}")
print(f"Person's metaclass: {type(Person)}")
print(f"type(Person) is type: {type(Person) is type}")

# The default metaclass is 'type'
print(f"\nDefault metaclass: {type}")
print(f"type.__class__: {type.__class__}")
print(f"type's MRO: {type.__mro__}")

In [ ]:
# How classes are created
print("\n=== CLASS CREATION PROCESS ===")

# Manual class creation using type()
def greet(self):
    return f"Hello, I'm {self.name}"

# Create class manually using type(name, bases, namespace)
ManualPerson = type('ManualPerson', (), {
    '__init__': lambda self, name: setattr(self, 'name', name),
    'greet': greet
})

manual_person = ManualPerson("Bob")
print(f"Manual class: {ManualPerson}")
print(f"Manual instance: {manual_person.greet()}")

# Compare with regular class definition
class RegularPerson:
    def __init__(self, name):
        self.name = name
    
    def greet(self):
        return f"Hello, I'm {self.name}"

regular_person = RegularPerson("Charlie")
print(f"\nRegular class: {RegularPerson}")
print(f"Regular instance: {regular_person.greet()}")

# Both are equivalent
print(f"\nBoth classes have same metaclass: {type(ManualPerson) is type(RegularPerson)}")
print(f"Both instances work the same way")

In [ ]:
# The __new__ and __init__ of metaclasses
print("\n=== METACLASS __new__ AND __init__ ===")

# Custom metaclass to demonstrate class creation
class VerboseMeta(type):
    """A metaclass that prints information during class creation."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        print(f"Meta.__new__ called with:")
        print(f"  name: {name}")
        print(f"  bases: {bases}")
        print(f"  namespace keys: {list(namespace.keys())}")
        
        # Call parent's __new__
        cls = super().__new__(mcs, name, bases, namespace, **kwargs)
        print(f"  created class: {cls}")
        return cls
    
    def __init__(cls, name, bases, namespace, **kwargs):
        print(f"Meta.__init__ called for {name}")
        super().__init__(name, bases, namespace, **kwargs)
        print(f"  class fully initialized: {cls}\n")

print("Creating class with VerboseMeta:")
class TestClass(metaclass=VerboseMeta):
    """A test class to demonstrate metaclass behavior."""
    
    def __init__(self, value):
        self.value = value
    
    def get_value(self):
        return self.value

print("Using the created class:")
instance = TestClass(42)
print(f"instance.get_value(): {instance.get_value()}")

## 🎯 Creating Custom Metaclasses

Custom metaclasses allow you to modify class behavior during creation.

In [ ]:
# Singleton metaclass
print("=== SINGLETON METACLASS ===")

class SingletonMeta(type):
    """Metaclass that ensures only one instance exists per class."""
    
    _instances = {}
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class DatabaseConnection(metaclass=SingletonMeta):
    """Database connection that should be a singleton."""
    
    def __init__(self, host, port):
        self.host = host
        self.port = port
        print(f"Created database connection to {host}:{port}")

# Test singleton behavior
conn1 = DatabaseConnection("localhost", 5432)
conn2 = DatabaseConnection("remotehost", 5432)

print(f"conn1 is conn2: {conn1 is conn2}")
print(f"conn1 == conn2: {conn1 == conn2}")
print(f"Same instance: {id(conn1) == id(conn2)}")

# Both should reference the same connection
print(f"conn1.host: {conn1.host}, conn2.host: {conn2.host}")

In [ ]:
# Validation metaclass
print("\n=== VALIDATION METACLASS ===")

class ValidationMeta(type):
    """Metaclass that validates class definitions."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        # Validate that all methods have docstrings
        for attr_name, attr_value in namespace.items():
            if callable(attr_value) and not attr_name.startswith('_'):
                if not hasattr(attr_value, '__doc__') or not attr_value.__doc__:
                    raise ValueError(f"Method {attr_name} in class {name} must have a docstring")
        
        # Validate that class has required attributes
        required_attrs = getattr(mcs, '_required_attrs', [])
        for attr in required_attrs:
            if attr not in namespace:
                raise ValueError(f"Class {name} must define attribute '{attr}'")
        
        return super().__new__(mcs, name, bases, namespace, **kwargs)

class StrictClass(metaclass=ValidationMeta):
    """A class that requires documentation for all methods."""
    
    def documented_method(self):
        """This method has proper documentation."""
        return "documented"
    
    def another_method(self):
        """This method is also documented."""
        return "also documented"

# This would fail - uncomment to test
# class BadClass(metaclass=ValidationMeta):
#     def undocumented_method(self):
#         return "no docs"  # Missing docstring!

print("StrictClass created successfully - all methods are documented!")
instance = StrictClass()
print(f"documented_method(): {instance.documented_method()}")
print(f"another_method(): {instance.another_method()}")

In [ ]:
# Automatic attribute management metaclass
print("\n=== AUTOMATIC ATTRIBUTE MANAGEMENT ===")

class AttributeTrackingMeta(type):
    """Metaclass that tracks all attributes created on instances."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        # Add attribute tracking to the class
        def __init__(self, *args, **kwargs):
            super(self.__class__, self).__init__(*args, **kwargs)
            self._tracked_attrs = set()
        
        def __setattr__(self, name, value):
            if not name.startswith('_') or name == '_tracked_attrs':
                self._tracked_attrs.add(name)
            super(self.__class__, self).__setattr__(name, value)
        
        def get_tracked_attributes(self):
            return sorted(self._tracked_attrs)
        
        # Add methods to namespace if they don't exist
        if '__init__' not in namespace:
            namespace['__init__'] = __init__
        if '__setattr__' not in namespace:
            namespace['__setattr__'] = __setattr__
        if 'get_tracked_attributes' not in namespace:
            namespace['get_tracked_attributes'] = get_tracked_attributes
        
        return super().__new__(mcs, name, bases, namespace, **kwargs)

class TrackedClass(metaclass=AttributeTrackingMeta):
    """A class that automatically tracks its attributes."""
    
    def __init__(self, name):
        self.name = name
        self.age = 25

# Test attribute tracking
obj = TrackedClass("Alice")
obj.city = "New York"
obj.job = "Engineer"

print(f"Object name: {obj.name}")
print(f"Tracked attributes: {obj.get_tracked_attributes()}")

# Add more attributes
obj.salary = 75000
obj.department = "R&D"

print(f"Updated tracked attributes: {obj.get_tracked_attributes()}")

## 🔗 Metaclass Inheritance and Composition

Metaclasses can inherit from each other and be combined using multiple inheritance.

In [ ]:
# Metaclass inheritance
print("=== METACLASS INHERITANCE ===")

class BaseMeta(type):
    """Base metaclass with common functionality."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        # Add a class registry
        if not hasattr(mcs, '_registry'):
            mcs._registry = {}
        mcs._registry[name] = mcs
        
        # Add creation timestamp
        namespace['_created_at'] = __import__('time').time()
        
        return super().__new__(mcs, name, bases, namespace, **kwargs)

class StrictMeta(BaseMeta):
    """Strict metaclass that inherits from BaseMeta."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        # Call parent first
        cls = super().__new__(mcs, name, bases, namespace, **kwargs)
        
        # Add strict validation
        for attr_name, attr_value in namespace.items():
            if attr_name.startswith('__') and attr_name.endswith('__'):
                continue  # Skip dunder methods
            if callable(attr_value) and not attr_name.startswith('_'):
                if not hasattr(attr_value, '__doc__') or not attr_value.__doc__:
                    raise ValueError(f"Method {attr_name} must have docstring")
        
        return cls

# Test metaclass inheritance
class BaseClass(metaclass=BaseMeta):
    """Base class using BaseMeta."""
    pass

class StrictClass(metaclass=StrictMeta):
    """Strict class using StrictMeta."""
    
    def method_one(self):
        """This method has documentation."""
        return "method_one"
    
    def method_two(self):
        """This method also has documentation."""
        return "method_two"

print(f"BaseClass created at: {BaseClass._created_at}")
print(f"StrictClass created at: {StrictClass._created_at}")
print(f"Registry: {BaseMeta._registry}")

# Test instances
base_obj = BaseClass()
strict_obj = StrictClass()

print(f"Base object: {base_obj}")
print(f"Strict object method_one: {strict_obj.method_one()}")

In [ ]:
# Metaclass composition with multiple inheritance
print("\n=== METACLASS COMPOSITION ===")

class LoggingMeta(type):
    """Metaclass that adds logging to all method calls."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        for attr_name, attr_value in namespace.items():
            if callable(attr_value) and not attr_name.startswith('_'):
                # Wrap the method with logging
                original_method = attr_value
                def logged_method(self, *args, **kwargs):
                    print(f"Calling {attr_name} with args={args}, kwargs={kwargs}")
                    result = original_method(self, *args, **kwargs)
                    print(f"{attr_name} returned: {result}")
                    return result
                namespace[attr_name] = logged_method
        
        return super().__new__(mcs, name, bases, namespace, **kwargs)

class ProfilingMeta(type):
    """Metaclass that adds profiling to all method calls."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        import time
        
        for attr_name, attr_value in namespace.items():
            if callable(attr_value) and not attr_name.startswith('_'):
                original_method = attr_value
                def profiled_method(self, *args, **kwargs):
                    start_time = time.time()
                    result = original_method(self, *args, **kwargs)
                    end_time = time.time()
                    print(f"{attr_name} took {(end_time - start_time)*1000:.2f}ms")
                    return result
                namespace[attr_name] = profiled_method
        
        return super().__new__(mcs, name, bases, namespace, **kwargs)

# Combine metaclasses (order matters!)
class LoggedProfiledMeta(LoggingMeta, ProfilingMeta):
    """Combined metaclass with both logging and profiling."""
    pass

class CombinedClass(metaclass=LoggedProfiledMeta):
    """Class that gets both logging and profiling."""
    
    def compute(self, n):
        """Compute something (simulated)."""
        import time
        time.sleep(0.1)  # Simulate work
        return n * n
    
    def process(self, data):
        """Process data (simulated)."""
        return f"Processed: {data.upper()}"

# Test combined functionality
obj = CombinedClass()
print("Testing combined metaclass (logging + profiling):")
result1 = obj.compute(5)
print(f"Compute result: {result1}\n")

result2 = obj.process("hello")
print(f"Process result: {result2}")

## 🔍 Practical Metaclass Use Cases

Real-world applications of metaclasses in Python frameworks and libraries.

In [ ]:
# Django-style ORM metaclass simulation
print("=== DJANGO-STYLE ORM SIMULATION ===")

class ModelMeta(type):
    """Metaclass for creating model classes with database field tracking."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        # Collect database fields
        fields = {}
        for attr_name, attr_value in namespace.items():
            if hasattr(attr_value, '_field_type'):
                fields[attr_name] = attr_value
        
        # Store fields on class
        namespace['_fields'] = fields
        namespace['_table_name'] = name.lower()
        
        # Create the class
        cls = super().__new__(mcs, name, bases, namespace, **kwargs)
        
        # Add table operations
        cls.objects = TableManager(cls)
        
        return cls

class TableManager:
    """Manager for table operations."""
    
    def __init__(self, model_class):
        self.model_class = model_class
    
    def create_table_sql(self):
        """Generate CREATE TABLE SQL."""
        fields_sql = []
        for field_name, field in self.model_class._fields.items():
            field_type = getattr(field, '_field_type', 'TEXT')
            nullable = "" if getattr(field, '_required', True) else "NULL"
            fields_sql.append(f"{field_name} {field_type} {nullable}")
        
        return f"CREATE TABLE {self.model_class._table_name} (\n  " + ",\n  ".join(fields_sql) + "\n);"
    
    def all(self):
        """Simulate querying all records."""
        return f"SELECT * FROM {self.model_class._table_name}"

# Field descriptors
class Field:
    """Base field descriptor."""
    
    def __init__(self, field_type, required=True):
        self._field_type = field_type
        self._required = required
    
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self._name, None)
    
    def __set__(self, instance, value):
        if self._required and value is None:
            raise ValueError(f"{self._name} is required")
        instance.__dict__[self._name] = value
    
    def __set_name__(self, owner, name):
        self._name = name

# Specific field types
def CharField(max_length=255, required=True):
    return Field(f"VARCHAR({max_length})", required)

def IntegerField(required=True):
    return Field("INTEGER", required)

# Model base class
class Model(metaclass=ModelMeta):
    """Base model class."""
    pass

# Define a model
class User(Model):
    """User model."""
    name = CharField(max_length=100)
    email = CharField(max_length=255)
    age = IntegerField()

# Test the ORM simulation
print(f"Table name: {User._table_name}")
print(f"Fields: {list(User._fields.keys())}")
print(f"\nCREATE TABLE SQL:")
print(User.objects.create_table_sql())
print(f"\nQuery all: {User.objects.all()}")

# Test model instance
user = User()
user.name = "John Doe"
user.email = "john@example.com"
user.age = 30

print(f"\nUser instance: name={user.name}, email={user.email}, age={user.age}")

In [ ]:
# API framework metaclass
print("\n=== API FRAMEWORK METACLASS ===")

class APIMeta(type):
    """Metaclass for creating API endpoint classes."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        # Collect endpoint methods
        endpoints = {}
        for attr_name, attr_value in namespace.items():
            if hasattr(attr_value, '_http_method'):
                method = getattr(attr_value, '_http_method')
                path = getattr(attr_value, '_path', f'/{attr_name}')
                endpoints[attr_name] = {
                    'method': method,
                    'path': path,
                    'handler': attr_value
                }
        
        # Store endpoints on class
        namespace['_endpoints'] = endpoints
        
        # Create API router
        namespace['router'] = APIRouter(endpoints)
        
        return super().__new__(mcs, name, bases, namespace, **kwargs)

class APIRouter:
    """Simple API router."""
    
    def __init__(self, endpoints):
        self.endpoints = endpoints
    
    def get_routes(self):
        """Get all routes."""
        return self.endpoints
    
    def handle_request(self, method, path):
        """Handle an API request."""
        for endpoint_name, endpoint_info in self.endpoints.items():
            if endpoint_info['method'] == method and endpoint_info['path'] == path:
                # Simulate calling the handler
                return f"{method} {path} -> {endpoint_name}()"
        return f"404 Not Found: {method} {path}"

# Decorators for HTTP methods
def get(path=None):
    def decorator(func):
        func._http_method = 'GET'
        func._path = path or f'/{func.__name__}'
        return func
    return decorator

def post(path=None):
    def decorator(func):
        func._http_method = 'POST'
        func._path = path or f'/{func.__name__}'
        return func
    return decorator

# API controller
class UserAPI(metaclass=APIMeta):
    """User API endpoints."""
    
    @get('/users')
    def get_users(self):
        """Get all users."""
        return "List of users"
    
    @get('/users/{id}')
    def get_user(self, user_id):
        """Get user by ID."""
        return f"User {user_id}"
    
    @post('/users')
    def create_user(self, user_data):
        """Create a new user."""
        return "User created"

# Test the API framework
print(f"API endpoints: {list(UserAPI._endpoints.keys())}")
print(f"Routes: {UserAPI.router.get_routes()}")

# Test requests
print("\nTesting API requests:")
print(UserAPI.router.handle_request('GET', '/users'))
print(UserAPI.router.handle_request('GET', '/users/123'))
print(UserAPI.router.handle_request('POST', '/users'))
print(UserAPI.router.handle_request('DELETE', '/users/123'))  # Should 404

## 🐛 Debugging Metaclasses

Debugging metaclass code can be challenging. Here are some techniques and best practices.

In [ ]:
# Debugging techniques for metaclasses
print("=== DEBUGGING METACLASSES ===")

# 1. Use print statements liberally
class DebuggableMeta(type):
    """Metaclass with built-in debugging."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        print(f"[DEBUG] Creating class: {name}")
        print(f"[DEBUG] Bases: {[base.__name__ for base in bases]}")
        print(f"[DEBUG] Namespace size: {len(namespace)}")
        
        # Check for common issues
        if '__init__' in namespace:
            print(f"[DEBUG] Found custom __init__")
        
        cls = super().__new__(mcs, name, bases, namespace, **kwargs)
        print(f"[DEBUG] Class created successfully: {cls}")
        return cls

print("Creating debuggable class:")
class DebugClass(metaclass=DebuggableMeta):
    """A class for debugging metaclass behavior."""
    
    def method(self):
        return "debug method"

# 2. Check MRO carefully
print(f"\nDebugClass MRO: {DebugClass.__mro__}")

# 3. Inspect metaclass relationships
print(f"DebugClass metaclass: {type(DebugClass)}")
print(f"DebugClass.__class__: {DebugClass.__class__}")

# 4. Test instance creation
try:
    instance = DebugClass()
    print(f"Instance created: {instance}")
    print(f"Method call: {instance.method()}")
except Exception as e:
    print(f"Error creating instance: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Common metaclass pitfalls and solutions
print("\n=== COMMON METACLASS PITFALLS ===")

# Pitfall 1: Modifying namespace dictionary during iteration
class BadMeta(type):
    """Example of a bad metaclass that modifies dict during iteration."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        # This can cause issues!
        for key in namespace:
            if key.startswith('_'):
                # Don't modify dict while iterating
                pass
        
        # Better approach: collect changes first
        new_attrs = {}
        for key, value in namespace.items():
            if key.startswith('temp_'):
                new_attrs[key.replace('temp_', 'final_')] = value
        
        namespace.update(new_attrs)
        return super().__new__(mcs, name, bases, namespace, **kwargs)

# Pitfall 2: Forgetting to call super()
class IncompleteMeta(type):
    """Metaclass that forgets to call super().__new__()."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        # This would break class creation!
        # return type(name, bases, namespace)  # Wrong!
        
        # Correct way:
        return super().__new__(mcs, name, bases, namespace, **kwargs)

# Pitfall 3: Infinite recursion in __setattr__
class ProblematicMeta(type):
    """Metaclass that can cause infinite recursion."""
    
    def __setattr__(cls, name, value):
        # This would cause infinite recursion!
        # cls.some_attr = value  # Wrong!
        
        # Correct way:
        super().__setattr__(name, value)

# Good practices
class GoodMeta(type):
    """Well-behaved metaclass following best practices."""
    
    def __new__(mcs, name, bases, namespace, **kwargs):
        # Validate inputs
        if not name:
            raise ValueError("Class name cannot be empty")
        
        # Make a copy of namespace to avoid issues
        namespace = dict(namespace)
        
        # Add debugging info in development
        if __debug__:
            namespace['_debug_info'] = {
                'created_by': mcs.__name__,
                'bases': [base.__name__ for base in bases],
                'timestamp': __import__('time').time()
            }
        
        return super().__new__(mcs, name, bases, namespace, **kwargs)

print("GoodMeta best practices:")
class GoodClass(metaclass=GoodMeta):
    """Class created with good metaclass practices."""
    pass

print(f"GoodClass debug info: {GoodClass._debug_info}")
print("\nPitfalls to avoid:")
print("✓ Don't modify namespace dict during iteration")
print("✓ Always call super().__new__()")
print("✓ Use super() for attribute access to avoid recursion")
print("✓ Validate inputs and handle edge cases")
print("✓ Add debugging info when developing metaclasses")

## 🏁 Summary and Best Practices

Congratulations! You've mastered Python metaclasses. Here's what you learned:

In [ ]:
# Metaclass Best Practices
print("=" * 60)
print("PYTHON METACLASSES MASTERY - COMPLETE GUIDE")
print("=" * 60)

metaclass_wisdom = {
    "Understanding Metaclasses": [
        "Metaclasses are classes that create classes",
        "Everything in Python is an object, including classes",
        "The default metaclass is type",
        "Classes are instances of their metaclass",
        "Metaclass __new__ controls class creation",
        "Metaclass __init__ initializes the created class"
    ],
    "When to Use Metaclasses": [
        "Implementing singletons or registries",
        "Enforcing coding standards or patterns",
        "Automatic code generation or modification",
        "Creating DSLs (Domain Specific Languages)",
        "Framework development (ORM, API frameworks)",
        "Complex inheritance control"
    ],
    "Metaclass Best Practices": [
        "Use metaclasses sparingly - they add complexity",
        "Document metaclass behavior clearly",
        "Test metaclasses thoroughly",
        "Consider alternatives (decorators, class decorators)",
        "Make metaclasses reusable when possible",
        "Handle edge cases and error conditions",
        "Use super() correctly to avoid issues"
    ],
    "Common Patterns": [
        "Singleton pattern for unique instances",
        "Validation metaclasses for code quality",
        "Attribute tracking and management",
        "Automatic method decoration",
        "Plugin systems and registries",
        "ORM and API framework foundations"
    ],
    "Debugging Techniques": [
        "Use print statements liberally in metaclasses",
        "Check MRO (Method Resolution Order) carefully",
        "Inspect metaclass relationships and inheritance",
        "Test class creation separately from instance creation",
        "Use try/except blocks for error isolation",
        "Add debugging attributes during development"
    ],
    "Alternatives to Metaclasses": [
        "Class decorators for simpler modifications",
        "__init_subclass__ for inheritance control",
        "Descriptors for attribute management",
        "Mixins for code reuse",
        "Factory functions for object creation",
        "Simple inheritance before reaching for metaclasses"
    ]
}

for category, wisdom in metaclass_wisdom.items():
    print(f"\n🧬 {category}:")
    for item in wisdom:
        print(f"   ✓ {item}")

print("\n" + "=" * 60)
print("🎯 METACLASS PHILOSOPHY")
print("=" * 60)
print("✓ Metaclasses are powerful but should be used judiciously")
print("✓ They enable deep customization of class behavior")
print("✓ Understanding metaclasses unlocks advanced Python patterns")
print("✓ Frameworks and libraries often use metaclasses internally")
print("✓ Simple problems don't need complex metaclass solutions")
print("✓ Metaclasses can make code harder to understand and debug")
print("✓ Master metaclasses to become a Python power user")
print("=" * 60)

## 📚 Additional Resources

- **Python Metaclasses Documentation**: https://docs.python.org/3/reference/datamodel.html#metaclasses
- **Metaclass Tutorial**: https://realpython.com/python-metaclasses/
- **Django Model Metaclass**: https://docs.djangoproject.com/en/stable/topics/db/models/
- **SQLAlchemy Metaclass Usage**: https://docs.sqlalchemy.org/
- **Advanced Python Patterns**: https://python-patterns.guide/

## 💡 Pro Tips

1. **Start Simple**: Understand basic class creation before metaclasses
2. **Use Sparingly**: Metaclasses add complexity - use only when necessary
3. **Document Well**: Metaclass behavior should be clearly documented
4. **Test Thoroughly**: Metaclasses need extensive testing
5. **Consider Alternatives**: Class decorators often suffice for simple needs
6. **Debug Carefully**: Use print statements and check MRO when debugging
7. **Follow Conventions**: Use super() and follow Python conventions
8. **Learn from Frameworks**: Study how Django, SQLAlchemy use metaclasses

---

**Congratulations!** 🎉 You've mastered Python metaclasses! You now understand one of Python's most powerful features. Metaclasses enable deep customization of class behavior and are essential for framework development. Keep exploring advanced Python patterns!

## Practice: Function and Descriptor Writing

These are advanced exercises. State the invariant your descriptor or metaclass enforces before implementing it.

### Q21 — Positive integer descriptor

Implement a descriptor `PositiveInt` for instance attributes. It must store values separately per instance, reject booleans and non-integers, reject values less than or equal to zero, and raise `AttributeError` when read before assignment.

```python
class Product:
    stock = PositiveInt()

item = Product()
item.stock = 3
assert item.stock == 3
```

### Q22 — Plugin registry metaclass

Implement a metaclass that registers concrete subclasses of `Plugin` by their non-empty `plugin_name` class attribute. Do not register the base class; reject duplicate names with `ValueError`.

```python
assert Plugin.registry["csv"].plugin_name == "csv"
```